# Newton's method check (Python)

Checks `thole.roots.newton` against analytic roots, mpmath, SciPy, and what the Lean theorems predict, and compares it with `brent`.
Start JupyterLab with `workshop run -- notebook`. After changing the Rust code, restart it.

In [ ]:
import math, sys
import mpmath
from scipy import optimize
import matplotlib.pyplot as plt
from thole.roots import brent, newton, ConvergenceError, NewtonRoot

XTOL, RTOL = 2e-12, 4 * sys.float_info.epsilon   # the defaults

## 1. A first call

`step` is the last step taken: `x` is the previous approximation minus `step`.

In [ ]:
r = newton(lambda x: x * x - 2, lambda x: 2 * x, 1.0)
print(r.x, r.step, r.iterations)
print("error      :", abs(r.x - math.sqrt(2)))
print("allowed    :", XTOL + RTOL * abs(r.x))
print("hex (compare with the Rust notebook):", r.x.hex())

## 2. Against mpmath (50 digits) and SciPy

With `xtol=1.48e-8, rtol=0`, the stopping test of `scipy.optimize.newton`, the two take the same steps and agree bit for bit.

In [ ]:
mpmath.mp.dps = 50
problems = [
    ("x^3 - x - 2", lambda x: x * x * x - x - 2,   lambda x: 3 * x * x - 1,          1.5, mpmath.findroot(lambda x: x**3 - x - 2, 1.5)),
    ("cos x - x",   lambda x: math.cos(x) - x,     lambda x: -math.sin(x) - 1,       0.5, mpmath.findroot(lambda x: mpmath.cos(x) - x, 0.5)),
    ("x e^x - 1",   lambda x: x * math.exp(x) - 1, lambda x: (x + 1) * math.exp(x),  0.5, mpmath.findroot(lambda x: x * mpmath.exp(x) - 1, 0.5)),
    ("x^2 - 2",     lambda x: x * x - 2,           lambda x: 2 * x,               1000.0, mpmath.sqrt(2)),
]
print(f"{'f':12}{'thole':>22}{'|thole - exact|':>17}{'steps':>7}{'scipy':>22}{'steps':>7}  same as scipy with its tolerance")
for name, f, df, x0, exact in problems:
    r = newton(f, df, x0)
    t = newton(f, df, x0, xtol=1.48e-8, rtol=0.0)
    s, info = optimize.newton(f, x0, fprime=df, full_output=True)
    print(f"{name:12}{r.x:22.16f}{float(abs(mpmath.mpf(r.x) - exact)):17.3e}{r.iterations:7d}{float(s):22.16f}{info.iterations:7d}  {t.x == s and t.iterations == info.iterations}")

## 3. What the Lean theorems predict

- `newtonStep_error`: one step squares the error, up to the factor `L / |f'(x)|`. For `x^2 - 2`, `L = 2` and `f'(x) = 2x`, so the error after a step is at most `e^2 / x`.
- `newton_error`: close enough to a simple root, every step at least halves the error.
- `newtonStep_exists_root`, `newton_error_of_done`: close to a simple root, a root lies within the last step of `x`.

In [ ]:
root = math.sqrt(2)
x = 3.0
print(f"{'x':>22}{'error':>12}{'bound e^2/x':>14}")
for _ in range(6):
    e = abs(x - root)
    nxt = x - (x * x - 2) / (2 * x)
    print(f"{nxt:22.16f}{abs(nxt - root):12.3e}{e * e / x:14.3e}")
    assert abs(nxt - root) <= e * e / x + 4 * sys.float_info.epsilon
    x = nxt

for xtol in [0.5, 0.1, 1e-2, 1e-4, 1e-8]:
    r = newton(lambda x: x * x - 2, lambda x: 2 * x, 2.0, xtol=xtol, rtol=0.0)
    print(f"xtol={xtol:<6g} x={r.x:.16f} error={abs(r.x - root):.3e} last step={abs(r.step):.3e} steps={r.iterations}")
    assert abs(r.x - root) <= abs(r.step) <= xtol

## 4. Convergence, next to Brent's method

Newton's method doubles the number of correct digits per step once it is close; it needs the derivative, and one more evaluation per step.

In [ ]:
def trace(f, df, x0, root, n):
    errors, x = [], x0
    for _ in range(n):
        x = x - f(x) / df(x)
        errors.append(abs(x - root))
    return errors

f, df = (lambda x: x * math.exp(x) - 1), (lambda x: (x + 1) * math.exp(x))
root = float(mpmath.findroot(lambda x: x * mpmath.exp(x) - 1, 0.5))
e = trace(f, df, 1.0, root, 6)
plt.semilogy(range(1, len(e) + 1), [max(v, 1e-17) for v in e], "o-", label="newton from 1.0")
plt.axhline(XTOL + RTOL * root, color="gray", linestyle="--", label="tolerance")
plt.xlabel("step"); plt.ylabel("|x - root|"); plt.legend(); plt.grid(True, which="both", alpha=0.3); plt.show()
print("brent on [0, 1]:", brent(f, 0, 1).iterations, "steps;  newton from 1.0:", newton(f, df, 1.0).iterations, "steps")

## 5. Where Newton's method fails

A double root: only linear convergence. A cycle: `x^3 - 2x + 2` from 0 alternates between 0 and 1. A run-away: `atan` from too far out. A short step far from any root: `x^2 + 1e-8` has none, but the guarantee needs the derivative to stay away from zero.

In [ ]:
r = newton(lambda x: (x - 1) ** 2, lambda x: 2 * (x - 1), 2.0, xtol=1e-6, rtol=0.0)
print("double root     :", r)
for name, call in {
    "cycle":         lambda: newton(lambda x: x**3 - 2 * x + 2, lambda x: 3 * x**2 - 2, 0.0),
    "run-away":      lambda: newton(math.atan, lambda x: 1 / (1 + x * x), 10.0),
    "zero slope":    lambda: newton(lambda x: x * x - 2, lambda x: 2 * x, 0.0),
    "no real root":  lambda: newton(lambda x: x * x + 1e-8, lambda x: 2 * x, 1.0, xtol=1e-3),
}.items():
    try:
        print(f"{name:16}:", call())
    except Exception as e:
        print(f"{name:16}: {type(e).__name__}: {e}")

## 6. Failures

In [ ]:
cases = {
    "infinite start":      lambda: newton(lambda x: x, lambda x: 1.0, math.inf),
    "negative tolerance":  lambda: newton(lambda x: x, lambda x: 1.0, 1.0, xtol=-1.0),
    "f returns NaN":       lambda: newton(lambda x: math.nan, lambda x: 1.0, 1.0),
    "fprime returns NaN":  lambda: newton(lambda x: x - 1, lambda x: math.nan, 0.0),
    "step overflows":      lambda: newton(lambda x: 1.0, lambda x: 1e-310, 0.0),
    "max_iter too small":  lambda: newton(math.cos, lambda x: -math.sin(x), 1.0, max_iter=2),
    "f raises":            lambda: newton(lambda x: 1 / 0, lambda x: 1.0, 1.0),
}
for name, call in cases.items():
    try:
        print(f"{name:22} -> returned {call()!r}")
    except Exception as e:
        print(f"{name:22} -> {type(e).__name__}: {e}")

## 7. Try your own

In [ ]:
r = newton(lambda x: math.sin(x) - 0.5, math.cos, 0.0)
r.x, math.pi / 6, r.iterations